In [1]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import torch
import faiss

In [2]:
import os

In [3]:
os.chdir('/home/rahul_goyal/product-debater-video-games/')

In [4]:
category_topics = {
    'Gaming Mice': ["Battery Life", "Ergonomics and Shape", "Build Quality", "Software and Drivers", "Switch and Click Quality", "Sensor Tracking", "Scroll Wheel"],
    'Gaming Keyboards': ["Typing and Switch Feel", "Build Quality", "RGB Lighting", "Software and Drivers", "Wrist Rest and Ergonomics", "Wireless Connectivity"],
    'Headsets': ["Sound and Audio Quality", "Microphone Quality", "Comfort and Earcups", "Battery Life", "Build Quality", "Connectivity and Lag"],
    'Controllers': ["Stick Drift and Thumbsticks", "Ergonomics and Grip", "Button and Trigger Feel", "Battery Life", "Build Quality", "D-Pad Quality"],
    'Consoles': ["Performance and Framerate", "Cooling and Fan Noise", "UI and Dashboard", "Storage Capacity", "Controller Quality", "Build Quality"],
    'Games': ["Story and Narrative", "Gameplay Mechanics", "Graphics and Visuals", "Bugs and Performance", "Replayability", "Multiplayer and Servers"]
}

In [5]:
print("Loading Models and Data...")
model = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"torch_dtype": torch.float16})
sentiment_analyzer = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest", device="cuda")

df_products = pd.read_parquet("data/meta_clean.parquet")
df_reviews = pd.read_parquet("data/semantic_metadata.parquet")
index = faiss.read_index("data/video_games_bge.index")

Loading Models and Data...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
pd.to_datetime(df_reviews['timestamp']).dt.year.value_counts().sort_index(ascending = False)

timestamp
2023    12773
2022    35301
2021    47799
2020    56450
2019    37328
2018    30138
2017    28897
2016    27344
2015    26112
2014    28339
2013    31950
2012    12092
2011    11792
2010     7168
2009     1876
2008      466
2007      205
2006       40
2005        2
2004        1
2003        7
Name: count, dtype: int64

In [7]:
df_reviews['year'] = pd.to_datetime(df_reviews['timestamp']).dt.year

In [8]:
df_reviews['year']

0         2020
1         2019
2         2018
3         2022
4         2021
          ... 
396075    2022
396076    2023
396077    2016
396078    2023
396079    2020
Name: year, Length: 396080, dtype: int32

In [9]:
# Store the final analytical data here
analytical_db = []

print("Starting Batch Aspect Sentiment Extraction (Time-Weighted)...")
for _, product in tqdm(df_products.iterrows(), total=len(df_products)):
    asin = product['parent_asin']
    category = product['leaf_category']
    
    # Skip if category is weird or missing
    if category not in category_topics:
        continue
        
    topics = category_topics[category]
    topic_vectors = model.encode(topics, normalize_embeddings=True, show_progress_bar=False)
    
    # Isolate product reviews
    product_indices = np.where(df_reviews['parent_asin'] == asin)[0]
    if len(product_indices) == 0:
        continue
        
    product_vectors = np.array([index.reconstruct(int(i)) for i in product_indices])
    similarities = np.dot(topic_vectors, product_vectors.T)
    
    for i, topic in enumerate(topics):
        topic_vector = topic_vectors[i].reshape(1, -1)
        
        # 1. Fetch the Top 200 based on raw vector similarity
        top_200_local_idx = np.argsort(similarities[i])[::-1][:200]
        
        # 2. Extract global indices, raw scores, and years
        candidate_global_idx = product_indices[top_200_local_idx]
        candidate_scores = similarities[i][top_200_local_idx]
        candidate_years = df_reviews.iloc[candidate_global_idx]['year'].values
        
        # 3. Build a temporary Dataframe to calculate the temporal penalty
        df_temporal = pd.DataFrame({
            'local_idx': top_200_local_idx,
            'raw_score': candidate_scores,
            'year': candidate_years
        })
        
        # 4. Apply Exponential Decay (5% penalty per year older than 2023)
        baseline_year = 2023
        df_temporal['years_old'] = baseline_year - df_temporal['year']
        df_temporal['years_old'] = df_temporal['years_old'].clip(lower=0) 
        df_temporal['weighted_score'] = df_temporal['raw_score'] * (0.95 ** df_temporal['years_old'])
        
        # 5. Sort by the new Weighted Score and slice the absolute best 100
        final_100_local_idx = df_temporal.sort_values(by='weighted_score', ascending=False).head(100)['local_idx'].values
        
        best_chunks_for_topic = []
        
        # 6. Proceed with the sliding window chunking on the optimized 100
        for local_idx in final_100_local_idx:
            global_idx = product_indices[local_idx]
            text = df_reviews.iloc[global_idx]['text']
            
            raw_sentences = [s.strip() + "." for s in re.split(r'[.!?]+', text) if len(s.strip()) > 10]
            if not raw_sentences: continue
            
            # Start with all 1-sentence chunks
            chunks = list(raw_sentences)
            
            # Add all 2-sentence sliding windows into the exact same pool
            if len(raw_sentences) > 1:
                chunks.extend([" ".join(raw_sentences[j:j+2]) for j in range(len(raw_sentences)-1)])
                
            # We let Cosine Similarity mathematically pick the absolute best context window.
            chunk_vectors = model.encode(chunks, normalize_embeddings=True, show_progress_bar=False, batch_size = 64)
            chunk_scores = np.dot(topic_vector, chunk_vectors.T)[0]
            best_chunks_for_topic.append(chunks[np.argmax(chunk_scores)])
            
        if not best_chunks_for_topic: continue
        
        # Batch analyze all chunks for this topic (truncation=True prevents token limit crashes)
        results = sentiment_analyzer(best_chunks_for_topic, batch_size=64, truncation=True, max_length=512)
        pos = sum(1 for r in results if r['label'] == 'positive')
        neu = sum(1 for r in results if r['label'] == 'neutral')
        neg = sum(1 for r in results if r['label'] == 'negative')
        
        # Save to our database
        analytical_db.append({
            "parent_asin": asin,
            "category": category,
            "aspect": topic,
            "positive": pos,
            "neutral": neu,
            "negative": neg,
            "total_analyzed": len(best_chunks_for_topic)
        })

Starting Batch Aspect Sentiment Extraction (Time-Weighted)...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 534/534 [3:07:43<00:00, 21.09s/it]


In [11]:
# Save the final analytical asset
df_analytics = pd.DataFrame(analytical_db)
df_analytics.to_parquet("data/aspect_sentiments.parquet", engine="pyarrow")